In [ ]:
import sys, os, json
from pathlib import Path

# Walk up from cwd until we find the project root (contains both src/ and data/)
_root = Path.cwd()
while _root != _root.parent:
    if (_root / "src").exists() and (_root / "data").exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root / "src"))

# Load API key from .env if present
try:
    from dotenv import load_dotenv
    load_dotenv(_root / ".env")
except ImportError:
    pass

from ingestor import load_locomo
from memory_writer import MemoryWriter
from memory_store import MemoryStore
from context_builder import ContextBuilder, RetrievalConfig
from baselines import FullHistoryBaseline, RollingSummaryBaseline, NaiveRAGBaseline
from evaluator import Evaluator, EvalSummary, ImportanceLearner

from openai import OpenAI
from sentence_transformers import SentenceTransformer

client = OpenAI()  # uses OPENAI_API_KEY env var
embedder = SentenceTransformer("all-MiniLM-L6-v2")

N_CONVS = 3        # number of conversations to evaluate
TOKEN_BUDGET = 500
print(f"Project root: {_root}")
print(f"Will evaluate on {N_CONVS} conversations, token budget={TOKEN_BUDGET}")

In [4]:
# Load data
conversations = load_locomo(_root / 'data' / 'locomo10.json')
eval_convs = [c for c in conversations[:N_CONVS] if c.qa_pairs]
total_qa = sum(len(c.qa_pairs) for c in eval_convs)
print(f'Evaluating on {len(eval_convs)} conversations, {total_qa} QA pairs')

NameError: name 'load_locomo' is not defined

In [ ]:
import time

# ── Build semantic cache memories ────────────
print('Building memory store...')
store = MemoryStore(collection_name='eval', embedder=embedder)
writer = MemoryWriter(
    client=client, embedder=embedder,
    window_size=4, stride=4,
    importance_threshold=0.3,
)

all_memories_by_session = {}
total_start = time.time()

for idx, conv in enumerate(eval_convs):
    n_windows = len(writer._make_windows(conv.turns))
    print(f'\n[{idx+1}/{len(eval_convs)}] session {conv.session_id} '
          f'({len(conv.turns)} turns, ~{n_windows} windows, ~{n_windows*2} API calls)')
    t0 = time.time()
    memories = writer.process_conversation(conv, verbose=True)
    elapsed = time.time() - t0
    all_memories_by_session[conv.session_id] = memories
    store.add_batch(memories)
    remaining = len(eval_convs) - idx - 1
    print(f'  → {len(memories)} memories stored  ({elapsed:.0f}s)'
          + (f'  ~{elapsed * remaining:.0f}s remaining' if remaining else ''))

print(f'\nDone. Total memories in store: {store.count()}  '
      f'({time.time()-total_start:.0f}s total)')

In [ ]:
# ── Initialize methods ────────────────────────
config = RetrievalConfig(token_budget=TOKEN_BUDGET)
builder = ContextBuilder(store=store, config=config)
evaluator = Evaluator(client=client, store=store, context_builder=builder)

full_history = FullHistoryBaseline(token_budget=TOKEN_BUDGET * 4)  # larger budget for fair comparison
rolling_summary = RollingSummaryBaseline(client=client, token_budget=TOKEN_BUDGET)
naive_rag = NaiveRAGBaseline(embedder=embedder, token_budget=TOKEN_BUDGET)

all_results = {}

In [ ]:
# ── Run: Semantic Cache ───────────────────────
print('Running: Semantic Cache...')
results_sc = evaluator.run_semantic_cache(
    eval_convs, all_memories_by_session, verbose=True
)
all_results['semantic_cache'] = results_sc
summary_sc = Evaluator.summarize(results_sc)
print(f'\nSemantic Cache: EM={summary_sc.exact_match_rate:.3f}, F1={summary_sc.avg_f1:.3f}, '
      f'tokens={summary_sc.avg_context_tokens:.0f}')

In [ ]:
# ── Run: Full History ─────────────────────────
print('Running: Full History...')
results_fh = evaluator.run_baseline(
    eval_convs,
    context_fn=lambda conv, q: full_history.build_context(conv, q),
    method_name='full_history',
    verbose=False,
)
all_results['full_history'] = results_fh
summary_fh = Evaluator.summarize(results_fh)
print(f'Full History: EM={summary_fh.exact_match_rate:.3f}, F1={summary_fh.avg_f1:.3f}, '
      f'tokens={summary_fh.avg_context_tokens:.0f}')

In [ ]:
# ── Run: Rolling Summary ──────────────────────
print('Running: Rolling Summary...')
results_rs = evaluator.run_baseline(
    eval_convs,
    context_fn=lambda conv, q: rolling_summary.build_context(conv, q),
    method_name='rolling_summary',
    verbose=False,
)
all_results['rolling_summary'] = results_rs
summary_rs = Evaluator.summarize(results_rs)
print(f'Rolling Summary: EM={summary_rs.exact_match_rate:.3f}, F1={summary_rs.avg_f1:.3f}, '
      f'tokens={summary_rs.avg_context_tokens:.0f}')

In [ ]:
# ── Run: Naive RAG ────────────────────────────
print('Running: Naive RAG...')
results_rag = evaluator.run_baseline(
    eval_convs,
    context_fn=lambda conv, q: naive_rag.build_context(conv, q),
    method_name='naive_rag',
    verbose=False,
)
all_results['naive_rag'] = results_rag
summary_rag = Evaluator.summarize(results_rag)
print(f'Naive RAG: EM={summary_rag.exact_match_rate:.3f}, F1={summary_rag.avg_f1:.3f}, '
      f'tokens={summary_rag.avg_context_tokens:.0f}')

In [ ]:
# ── Save all results ──────────────────────────
os.makedirs(_root / 'results', exist_ok=True)

for method, results in all_results.items():
    Evaluator.save_results(results, str(_root / 'results' / f'{method}_results.json'))

# Save summaries
summaries_dict = {s.method: s.to_dict() for s in summaries}
with open(_root / 'results' / 'eval_results.json', 'w') as f:
    json.dump(summaries_dict, f, indent=2)
print('All results saved.')

In [ ]:
# ── Collect summaries and save all results ────
summaries = [summary_sc, summary_fh, summary_rs, summary_rag]

os.makedirs(_root / "results", exist_ok=True)

for method, results in all_results.items():
    Evaluator.save_results(results, str(_root / "results" / f"{method}_results.json"))

summaries_dict = {s.method: s.to_dict() for s in summaries}
with open(_root / "results" / "eval_results.json", "w") as f:
    json.dump(summaries_dict, f, indent=2)
print("All results saved to results/")

In [ ]:
# ── Token efficiency: F1 per token ────────────
print('\n=== Token Efficiency (F1 per token) ===')
for s in summaries:
    if s.avg_context_tokens and s.avg_context_tokens > 0:
        efficiency = s.avg_f1 / s.avg_context_tokens * 100
        print(f'  {s.method}: {efficiency:.4f} F1 per 100 tokens')

In [ ]:
# ── Visualizations ────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

methods = [s.method for s in summaries]
em_scores = [s.exact_match_rate for s in summaries]
f1_scores = [s.avg_f1 for s in summaries]
token_counts = [s.avg_context_tokens for s in summaries]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
x = np.arange(len(methods))
colors = ["#2196F3", "#FF5722", "#4CAF50", "#FF9800"]

axes[0].bar(x, em_scores, color=colors)
axes[0].set_xticks(x)
axes[0].set_xticklabels([m.replace("_", "\n") for m in methods], fontsize=9)
axes[0].set_title("Exact Match Rate")
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("Score")

axes[1].bar(x, f1_scores, color=colors)
axes[1].set_xticks(x)
axes[1].set_xticklabels([m.replace("_", "\n") for m in methods], fontsize=9)
axes[1].set_title("Token F1")
axes[1].set_ylim(0, 1)

axes[2].bar(x, token_counts, color=colors)
axes[2].set_xticks(x)
axes[2].set_xticklabels([m.replace("_", "\n") for m in methods], fontsize=9)
axes[2].set_title("Avg Context Tokens")
axes[2].set_ylabel("Tokens")

plt.suptitle("Method Comparison on LoCoMo QA", fontsize=13, fontweight="bold")
plt.tight_layout()
chart_path = str(_root / "results" / "comparison_chart.png")
plt.savefig(chart_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Chart saved to {chart_path}")

In [ ]:
# ── RLVR Reward Distribution ──────────────────
rewards = [r.reward for r in results_sc if r.reward is not None]

if rewards:
    reward_counts = {1.0: 0, 0.0: 0, -0.5: 0, -1.0: 0}
    for r in rewards:
        if r in reward_counts:
            reward_counts[r] += 1
    
    print('RLVR Reward Distribution (Semantic Cache):')
    for val, count in sorted(reward_counts.items(), reverse=True):
        pct = count / len(rewards) * 100
        label = {1.0: 'Correct + retrieved', 0.0: 'Other',
                 -0.5: 'Missing retrieval', -1.0: 'Not in memory'}
        print(f'  {val:+.1f} ({label[val]}): {count} ({pct:.1f}%)')
    print(f'  Mean reward: {sum(rewards)/len(rewards):.3f}')

In [ ]:
# ── Ablation: importance weight ───────────────
print('\n=== Ablation: β (importance weight) ===')
betas = [0.0, 0.1, 0.3, 0.5]
ablation_results = []

for beta in betas:
    cfg = RetrievalConfig(alpha=0.5, beta=beta, gamma=0.1, lam=0.1, token_budget=TOKEN_BUDGET)
    ab_builder = ContextBuilder(store=store, config=cfg)
    ab_eval = Evaluator(client=client, store=store, context_builder=ab_builder)
    ab_res = ab_eval.run_semantic_cache(eval_convs, all_memories_by_session)
    ab_sum = Evaluator.summarize(ab_res)
    ablation_results.append((beta, ab_sum.exact_match_rate, ab_sum.avg_f1))
    print(f'  β={beta:.1f}: EM={ab_sum.exact_match_rate:.3f}, F1={ab_sum.avg_f1:.3f}')